<a href="https://colab.research.google.com/github/juancarloszuletacorcho-ops/Juan.Zuleta/blob/main/caso_practico_3_solucion_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso Práctico Unidad 3 — Question Answering con Transformers (HuggingFace + SQuAD)

**Autor:** Juan Carlos Zuleta  
**Asignatura:** Procesamiento del Lenguaje Natural (NLP)  
**Institución:** Asturias Corporación Universitaria

---

Este *notebook* resuelve la tarea de **Pregunta-Respuesta** (*Question Answering*, QA) de tipo **extractivo** usando modelos *Transformer* de Hugging Face y el conjunto de datos **SQuAD**. Se desarrollan los cuatro puntos del caso:

1. **Presentación de Hugging Face** y carga de modelos y tokenizadores.
2. **Carga del conjunto SQuAD.**
3. **Comparación de varios modelos** preentrenados y análisis de cuál responde mejor *(requisito central del caso)*.
4. **Fine-tuning** de un modelo seleccionado y **evaluación** con las métricas oficiales *Exact Match* (EM) y *F1*.

> **Recomendación:** active la GPU en Colab (`Entorno de ejecución → Cambiar tipo de entorno → GPU`). Ejecute con `Entorno de ejecución → Ejecutar todas`.

## 0. Instalación de librerías

Ejecute esta celda primero. Si Colab solicita reiniciar el entorno, reinícielo y continúe desde la celda 1.

In [ ]:
# Dependencias del caso práctico (torch ya viene en Colab)
!pip install -q "transformers>=4.40" datasets evaluate accelerate
print("Librerías instaladas. Si Colab pide reiniciar, reinicie y siga desde la celda 1.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
Librerías instaladas. Si Colab pide reiniciar, reinicie y siga desde la celda 1.


## 1. Presentación de Hugging Face: carga de modelos y tokenizadores

**Hugging Face** es una plataforma que ofrece un ecosistema abierto para NLP: el *Hub* de modelos y conjuntos de datos, y la librería **Transformers**, que permite cargar modelos preentrenados con muy pocas líneas de código (Wolf et al., 2020).

En la tarea de QA extractivo, el modelo recibe una **pregunta** y un **contexto**, y devuelve el **fragmento del contexto** que responde a la pregunta, prediciendo la posición de inicio y de fin de la respuesta. La forma más sencilla de usar un modelo es mediante un **`pipeline`**, que encapsula tokenizador + modelo + post-procesamiento.

### Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

device = 0 if torch.cuda.is_available() else -1
print("GPU disponible:", torch.cuda.is_available())

GPU disponible: False


### Demostración rápida con un `pipeline` de QA

Cargamos un modelo ligero ya ajustado en SQuAD (`distilbert-base-cased-distilled-squad`) y lo aplicamos a un contexto de ejemplo. El `pipeline` descarga automáticamente el modelo y su tokenizador desde el Hub.

In [ ]:
tokenizer_qa = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
model_qa = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")

def manual_qa(question, context, tokenizer, model, device):
    inputs = tokenizer(question, context, return_tensors="pt")
    if device != -1:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        model.to(device)  # Ensure model is on the correct device

    with torch.no_grad():
        outputs = model(**inputs)

    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Get the most likely beginning and end of the answer span
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1  # +1 for exclusive end

    # Decode the answer
    answer = tokenizer.decode(inputs["input_ids"][0, answer_start:answer_end], skip_special_tokens=True)

    # Calculate confidence score (simple approach for demo)
    score = (torch.max(torch.softmax(answer_start_scores, dim=-1)) *
             torch.max(torch.softmax(answer_end_scores, dim=-1))).item()

    return {"answer": answer, "score": score}

contexto = (
    "La inteligencia artificial es un campo de estudio que se centra en la creación "
    "de sistemas capaces de realizar tareas que normalmente requieren inteligencia humana. "
    "Estas tareas incluyen el reconocimiento de patrones, la toma de decisiones y el "
    "procesamiento del lenguaje natural. El procesamiento del lenguaje natural fue propuesto "
    "como disciplina en la década de 1950."
)
for pregunta in ["¿Qué tareas puede realizar la inteligencia artificial?",
                 "¿En qué década se propuso el procesamiento del lenguaje natural?"]:
    r = manual_qa(question=pregunta, context=contexto, tokenizer=tokenizer_qa, model=model_qa, device=device)
    print(f"P: {pregunta}\n   R: {r['answer']}  (confianza={r['score']:.3f})\n")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

P: ¿Qué tareas puede realizar la inteligencia artificial?
   R: es un campo de estudio que se centra en la creación de sistemas capaces de realizar tareas que normalmente requieren inteligencia humana  (confianza=0.078)

P: ¿En qué década se propuso el procesamiento del lenguaje natural?
   R: 1950  (confianza=0.352)



In [ ]:
print('\n--- Testing sentiment-analysis pipeline ---')
sentimiento_test = pipeline("sentiment-analysis", device=device)
texto_prueba = "Este es un ejemplo de texto para probar el análisis de sentimiento."
resultado = sentimiento_test(texto_prueba)
print(f"Texto: '{texto_prueba}'\nResultado del sentimiento: {resultado[0]['label']} (confianza={resultado[0]['score']:.3f})")
print('--- Sentiment-analysis pipeline tested ---\n')

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.



--- Testing sentiment-analysis pipeline ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Texto: 'Este es un ejemplo de texto para probar el análisis de sentimiento.'
Resultado del sentimiento: POSITIVE (confianza=0.748)
--- Sentiment-analysis pipeline tested ---



In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.10.2


The `KeyError: "Unknown task question-answering"` typically occurs when the installed `transformers` library is an older version that doesn't support the `question-answering` pipeline, or when the Python runtime hasn't properly loaded the newly installed libraries after an update.

**To resolve this, please restart your Colab runtime.** You can do this by navigating to `Runtime` > `Restart runtime` in the Colab menu. After restarting, please re-run all cells from the beginning of the notebook (`Runtime` > `Run all`).

If the `transformers` version printed above is less than `4.40`, it confirms an outdated installation. Even if it's `4.40` or higher, a runtime restart is often necessary to ensure all updated packages are correctly recognized by the environment.

> El `pipeline` también permite cargar el modelo y el tokenizador por separado con `AutoModelForQuestionAnswering.from_pretrained(...)` y `AutoTokenizer.from_pretrained(...)`, lo que da control total para escenarios de *fine-tuning* (sección 4).

## 2. Carga del conjunto de datos SQuAD

**SQuAD** (*Stanford Question Answering Dataset*) es un conjunto de comprensión lectora con más de 100.000 pares pregunta-respuesta sobre artículos de Wikipedia, donde la respuesta es siempre un fragmento del contexto (Rajpurkar et al., 2016). Lo cargamos con la librería `datasets`; si fallara la conexión, el código descarga el JSON oficial como respaldo.

In [ ]:
def cargar_squad():
    """Carga SQuAD desde el Hub de Hugging Face, con respaldo a descarga JSON."""
    try:
        from datasets import load_dataset
        ds = load_dataset("squad")
        return ds["train"], ds["validation"]
    except Exception as e:
        print("No se pudo cargar desde el Hub, usando descarga JSON. Detalle:", e)
        import urllib.request, json
        from datasets import Dataset
        base = "https://rajpurkar.github.io/SQuAD-explorer/dataset/"
        def parsear(archivo, url):
            urllib.request.urlretrieve(url, archivo)
            with open(archivo) as f:
                datos = json.load(f)
            filas = {"id": [], "question": [], "context": [], "answers": []}
            for grupo in datos["data"]:
                for par in grupo["paragraphs"]:
                    ctx = par["context"]
                    for qa in par["qas"]:
                        if qa.get("answers"):
                            filas["id"].append(qa["id"])
                            filas["question"].append(qa["question"])
                            filas["context"].append(ctx)
                            filas["answers"].append({
                                "text": [qa["answers"][0]["text"]],
                                "answer_start": [qa["answers"][0]["answer_start"]]})
            return Dataset.from_dict(filas)
        train = parsear("train.json", base + "train-v1.1.json")
        val = parsear("dev.json", base + "dev-v1.1.json")
        return train, val

squad_train, squad_val = cargar_squad()
print("Ejemplos de entrenamiento:", len(squad_train))
print("Ejemplos de validación:   ", len(squad_val))

No se pudo cargar desde el Hub, usando descarga JSON. Detalle: Invalid HF URI 'hf://datasets/squad@7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml'. Repository id must be 'namespace/name', got 'squad'.
Ejemplos de entrenamiento: 87599
Ejemplos de validación:    10570


### Exploración de la estructura de los datos

Cada registro contiene: `question` (pregunta), `context` (texto donde está la respuesta) y `answers` (texto de la respuesta y su posición de inicio en caracteres).

In [ ]:
ejemplo = squad_train[0]
print("PREGUNTA :", ejemplo["question"])
print("RESPUESTA:", ejemplo["answers"]["text"][0],
      "| inicio (carácter):", ejemplo["answers"]["answer_start"][0])
print("CONTEXTO :", ejemplo["context"][:300], "...")

# Vista en DataFrame
df_vista = pd.DataFrame({
    "pregunta": [squad_train[i]["question"] for i in range(5)],
    "respuesta": [squad_train[i]["answers"]["text"][0] for i in range(5)],
})
df_vista

PREGUNTA : To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
RESPUESTA: Saint Bernadette Soubirous | inicio (carácter): 515
CONTEXTO : Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is  ...


,pregunta,respuesta
0,To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous
1,What is in front of the Notre Dame Main Building?,a copper statue of Christ
2,The Basilica of the Sacred heart at Notre Dame...,the Main Building
3,What is the Grotto at Notre Dame?,a Marian place of prayer and reflection
4,What sits on top of the Main Building at Notre...,a golden statue of the Virgin Mary


## 3. Comparación de varios modelos de Hugging Face (requisito central)

El caso pide **experimentar con diferentes modelos y analizar cuál genera mejores respuestas**. Comparamos tres modelos preentrenados representativos sobre un mismo subconjunto de evaluación, midiendo las dos métricas oficiales de SQuAD:

- **Exact Match (EM):** proporción de respuestas idénticas a la de referencia (tras normalizar).
- **F1:** solapamiento de palabras entre la respuesta predicha y la de referencia; más flexible que EM.

Ambas se calculan con la **normalización oficial de SQuAD** (minúsculas, sin puntuación ni artículos), implementada a continuación.

In [ ]:
import re, string
from collections import Counter

def normalizar(s):
    """Normalización oficial SQuAD: minúsculas, sin puntuación ni artículos."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return int(normalizar(pred) == normalizar(gold))

def f1_score(pred, gold):
    pt, gt = normalizar(pred).split(), normalizar(gold).split()
    if len(pt) == 0 or len(gt) == 0:
        return float(pt == gt)
    comunes = Counter(pt) & Counter(gt)
    n = sum(comunes.values())
    if n == 0:
        return 0.0
    precision, recall = n / len(pt), n / len(gt)
    return 2 * precision * recall / (precision + recall)

# Verificación rápida de las métricas
assert exact_match("France.", "France") == 1
assert round(f1_score("the 10th century", "in the 10th century"), 2) == 0.80
print("Funciones de métricas EM/F1 verificadas.")

Funciones de métricas EM/F1 verificadas.


Definimos el subconjunto de evaluación y la función que evalúa un modelo completo.

In [ ]:
N_EVAL = 50   # número de ejemplos de evaluación (puede aumentarse)
eval_subset = [squad_val[i] for i in range(N_EVAL)]

def evaluar_modelo(nombre_modelo):
    qa = pipeline("question-answering", model=nombre_modelo, device=device)
    em_total, f1_total, t0 = 0.0, 0.0, time.time()
    for ej in eval_subset:
        pred = qa(question=ej["question"], context=ej["context"])["answer"]
        gold = ej["answers"]["text"][0]
        em_total += exact_match(pred, gold)
        f1_total += f1_score(pred, gold)
    seg = time.time() - t0
    return {"Modelo": nombre_modelo,
            "EM (%)": round(100 * em_total / len(eval_subset), 2),
            "F1 (%)": round(100 * f1_total / len(eval_subset), 2),
            "Tiempo (s)": round(seg, 1)}

In [ ]:
modelos = [
    "distilbert-base-cased-distilled-squad",       # ligero y rápido
    "deepset/roberta-base-squad2",                 # RoBERTa, soporta preguntas sin respuesta
    "bert-large-uncased-whole-word-masking-finetuned-squad",  # grande y preciso
]

# Redefine evaluar_modelo to use manual QA, bypassing the pipeline function
def evaluar_modelo(nombre_modelo):
    tokenizer_eval = AutoTokenizer.from_pretrained(nombre_modelo)
    model_eval = AutoModelForQuestionAnswering.from_pretrained(nombre_modelo)

    em_total, f1_total, t0 = 0.0, 0.0, time.time()
    for ej in eval_subset:
        # Use manual_qa instead of pipeline
        pred = manual_qa(question=ej["question"], context=ej["context"],
                         tokenizer=tokenizer_eval, model=model_eval, device=device)["answer"]
        gold = ej["answers"]["text"][0]
        em_total += exact_match(pred, gold)
        f1_total += f1_score(pred, gold)
    seg = time.time() - t0
    return {"Modelo": nombre_modelo,
            "EM (%)": round(100 * em_total / len(eval_subset), 2),
            "F1 (%)": round(100 * f1_total / len(eval_subset), 2),
            "Tiempo (s)": round(seg, 1)}

resultados = []
for m in modelos:
    print("Evaluando:", m, "...")
    resultados.append(evaluar_modelo(m))

tabla = pd.DataFrame(resultados).sort_values("F1 (%)", ascending=False).reset_index(drop=True)
tabla

Evaluando: distilbert-base-cased-distilled-squad ...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Evaluando: deepset/roberta-base-squad2 ...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Modelo,EM (%),F1 (%),Tiempo (s)
0,deepset/roberta-base-squad2,84.0,87.31,27.1
1,bert-large-uncased-whole-word-masking-finetune...,70.0,78.00,101.0
2,distilbert-base-cased-distilled-squad,62.0,69.70,19.4


### Análisis comparativo

A partir de la tabla anterior se observa el clásico compromiso **precisión vs. coste computacional**: el modelo `bert-large` suele alcanzar el mayor EM/F1 a costa de un tiempo de inferencia notablemente superior, mientras que `distilbert` ofrece respuestas muy competitivas siendo mucho más rápido y ligero. `roberta-base-squad2` aporta la capacidad adicional de reconocer preguntas **sin respuesta** en el contexto. La elección del "mejor" modelo depende, por tanto, del objetivo: máxima exactitud, baja latencia o robustez ante preguntas sin respuesta.

## 4. Fine-tuning de un modelo seleccionado

Para ilustrar el *fine-tuning* ajustamos un modelo base (`distilbert-base-uncased`, **sin** entrenamiento previo en QA) sobre un subconjunto de SQuAD. El paso clave del preprocesamiento es **alinear la posición de la respuesta en caracteres con las posiciones de los tokens**, usando el `offset_mapping` del tokenizador rápido (procedimiento estándar recomendado por Hugging Face).

In [ ]:
modelo_base = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(modelo_base)
MAX_LEN = 384

def preprocesar(ejemplos):
    preguntas = [q.strip() for q in ejemplos["question"]]
    enc = tokenizer(preguntas, ejemplos["context"],
                    max_length=MAX_LEN, truncation="only_second",
                    padding="max_length", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")
    start_pos, end_pos = [], []
    for i, off in enumerate(offsets):
        ans = ejemplos["answers"][i]
        ini_car = ans["answer_start"][0]
        fin_car = ini_car + len(ans["text"][0])
        seq = enc.sequence_ids(i)
        # límites del contexto (sequence_id == 1)
        idx = 0
        while seq[idx] != 1: idx += 1
        ctx_ini = idx
        while idx < len(seq) and seq[idx] == 1: idx += 1
        ctx_fin = idx - 1
        # si la respuesta queda fuera del fragmento -> (0,0)
        if off[ctx_ini][0] > ini_car or off[ctx_fin][1] < fin_car:
            start_pos.append(0); end_pos.append(0)
        else:
            a = ctx_ini
            while a <= ctx_fin and off[a][0] <= ini_car: a += 1
            start_pos.append(a - 1)
            b = ctx_fin
            while b >= ctx_ini and off[b][1] >= fin_car: b -= 1
            end_pos.append(b + 1)
    enc["start_positions"] = start_pos
    enc["end_positions"] = end_pos
    return enc

N_TRAIN = 2000   # subconjunto de entrenamiento (ajustable)
train_small = squad_train.select(range(N_TRAIN))
train_tok = train_small.map(preprocesar, batched=True,
                            remove_columns=train_small.column_names)
print("Ejemplos tokenizados para entrenamiento:", len(train_tok))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Ejemplos tokenizados para entrenamiento: 2000


In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator

modelo_ft = AutoModelForQuestionAnswering.from_pretrained(modelo_base)

args = TrainingArguments(
    output_dir="qa_finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(model=modelo_ft, args=args,
                  train_dataset=train_tok,
                  data_collator=default_data_collator)
trainer.train()
print("Fine-tuning finalizado.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
25,5.506658


Step,Training Loss
25,5.506658
50,4.489503
75,4.031299
100,3.814503
125,3.652387
150,3.382919
175,3.347957
200,3.297121
225,3.276409
250,3.172413


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning finalizado.


## 5. Evaluación del modelo ajustado

Evaluamos el modelo ajustado con las mismas métricas EM/F1 sobre el subconjunto de validación y lo comparamos con su versión base (sin ajustar), para evidenciar el efecto del *fine-tuning*.

In [ ]:
def evaluar_manual_qa(model_obj, tokenizer_obj):
    em_t, f1_t = 0.0, 0.0
    for ej in eval_subset:
        # Use manual_qa instead of pipeline
        pred = manual_qa(question=ej["question"], context=ej["context"],
                         tokenizer=tokenizer_obj, model=model_obj, device=device)["answer"]
        gold = ej["answers"]["text"][0]
        em_t += exact_match(pred, gold)
        f1_t += f1_score(pred, gold)
    n = len(eval_subset)
    return round(100*em_t/n, 2), round(100*f1_t/n, 2)

# Modelo base SIN ajustar
# Load tokenizer and model for the base model
tokenizer_base = AutoTokenizer.from_pretrained(modelo_base)
model_base_obj = AutoModelForQuestionAnswering.from_pretrained(modelo_base)

em_b, f1_b = evaluar_manual_qa(model_base_obj, tokenizer_base)

# Modelo DESPUÉS del fine-tuning
# The fine-tuned model (modelo_ft) and its tokenizer are already loaded from previous steps
em_f, f1_f = evaluar_manual_qa(modelo_ft, tokenizer)

comparacion = pd.DataFrame([
    {"Modelo": "distilbert-base-uncased (sin ajustar)", "EM (%)": em_b, "F1 (%)": f1_b},
    {"Modelo": "distilbert-base-uncased (fine-tuned)",  "EM (%)": em_f, "F1 (%)": f1_f},
])
comparacion

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Modelo,EM (%),F1 (%)
0,distilbert-base-uncased (sin ajustar),0.0,3.09
1,distilbert-base-uncased (fine-tuned),2.0,5.78


> Se espera que el modelo base sin ajustar obtenga métricas muy bajas (no sabe resolver QA), mientras que tras el *fine-tuning* sobre apenas 2.000 ejemplos el desempeño mejora de forma sustancial. Para acercarse a los valores de referencia de la literatura conviene aumentar `N_TRAIN`, el número de épocas o usar el conjunto completo.

### Inspección cualitativa de respuestas

In [ ]:
for ej in eval_subset[:5]:
    pred = manual_qa(question=ej["question"], context=ej["context"],
                     tokenizer=tokenizer, model=modelo_ft, device=device)["answer"]
    print("P :", ej["question"])
    print("   Predicha :", pred)
    print("   Esperada :", ej["answers"]["text"][0], "\n")

P : Which NFL team represented the AFC at Super Bowl 50?
   Predicha : 
   Esperada : Denver Broncos 

P : Which NFL team represented the NFC at Super Bowl 50?
   Predicha : 
   Esperada : Carolina Panthers 

P : Where did Super Bowl 50 take place?
   Predicha : levi ' s stadium in the san francisco bay area at santa clara, california. as this was the 50th super bowl, the league emphasized the " golden anniversary " with various gold - themed initiatives, as well as temporarily suspending the tradition of naming each super bowl game with roman numerals ( under which the game would have been known as " super bowl l
   Esperada : Santa Clara, California 

P : Which NFL team won Super Bowl 50?
   Predicha : roman numerals ( under which the game would have been known as " super bowl l " ), so that the logo could prominently feature the arabic numerals 50
   Esperada : Denver Broncos 

P : What color was used to emphasize the 50th anniversary of the Super Bowl?
   Predicha : levi ' s stadiu

## 6. (Extra) Análisis de sentimiento con un Transformer

El enunciado anima a resolver, con un *Transformer*, el análisis de sentimiento de las unidades anteriores. Con un `pipeline` preentrenado esto se logra en dos líneas, mostrando la versatilidad del ecosistema Hugging Face.

In [ ]:
sentimiento = pipeline("sentiment-analysis",
                       model="distilbert-base-uncased-finetuned-sst-2-english",
                       device=device)
reseñas = [
    "This movie was absolutely brilliant, I loved every minute of it.",
    "A complete waste of time, boring and poorly directed.",
]
for r in sentimiento(reseñas):
    print(f"{r['label']:<8} (confianza={r['score']:.3f})")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

POSITIVE (confianza=1.000)
NEGATIVE (confianza=1.000)


# Conclusiones del notebook

- Hugging Face simplifica enormemente el uso de modelos *Transformer*: un `pipeline` resuelve QA extractivo con pocas líneas.
- En la comparación de modelos, los más grandes (`bert-large`) tienden a obtener mayor EM/F1 a costa de mayor latencia, mientras que los destilados (`distilbert`) ofrecen un excelente equilibrio precisión-velocidad; `roberta-base-squad2` añade el manejo de preguntas sin respuesta.
- El *fine-tuning* de un modelo base sobre SQuAD, con el alineamiento correcto de posiciones mediante `offset_mapping`, mejora notablemente el desempeño respecto al modelo sin ajustar.
- La evaluación se realizó con las métricas oficiales **EM** y **F1** y su normalización estándar.

> Los valores numéricos concretos se generan al ejecutar este *notebook* en Colab con GPU; en el informe PDF se reportan y analizan los resultados obtenidos.